# Enriquecimento de Metadados ICCRTS — Keywords & Referências APA (v3 → v4)

Este notebook enriquece o arquivo `metadados_iccrts_v3.xlsx` com dois novos campos:

1. **`LLM_KEYWORDS`** — Keywords geradas pela LLM para artigos que não possuem keywords registradas (campo `KEYWORDS` vazio). A LLM lê o arquivo `.md` original do Drive para gerar as keywords.
2. **`LLM_REFERENCES`** — Referências da coluna `REFERENCES` reformatadas para o padrão **APA 7ª edição** pela LLM. Não é necessário reler os arquivos `.md` — os dados já estão na planilha v3.

**Estratégia de economia:**
- Pass 1 (Keywords): ~359 chamadas à API (apenas artigos com `KEYWORDS` vazio)
- Pass 2 (Referências): ~450 chamadas à API (apenas artigos com `REFERENCES` não vazio)
- Cada pass tem seu próprio checkpoint para retomada independente

**Saída:** `metadados_iccrts_v4.xlsx`

**Modelo:** `openai/gpt-oss-120b` via Groq API

> Notebook projetado para rodar no **Google Colab** com kernel Python.

In [1]:
# Cell 2: Install dependencies
!pip install groq pandas openpyxl tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 3.0 MB/s eta 0:00:00a 0:00:01


In [2]:
# Cell 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Cell 4: API Key setup (secure input via getpass)
import getpass
import os

print("Autenticação necessária:")
os.environ['GROQ_API_KEY'] = getpass.getpass('Cole sua GROQ_API_KEY: ')
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')

if not GROQ_API_KEY:
    print("ERRO: Chave não encontrada!")
else:
    print("Chave carregada com sucesso.")

Autenticação necessária:
Chave carregada com sucesso.


In [4]:
# Cell 5: Configuration constants

# Base path on Google Drive
BASE_PATH = '/content/drive/My Drive/Mestrado/Mestrado PPCA/Dissertaçao/Artigos_C2_IC2Institute/ICCRTS'

# Input and output files
INPUT_FILE  = BASE_PATH + '/metadados_iccrts_v3.xlsx'
OUTPUT_FILE = BASE_PATH + '/metadados_iccrts_v4.xlsx'

# Checkpoint files (one per pass)
CHECKPOINT_KEYWORDS = BASE_PATH + '/checkpoint_keywords_v4.json'
CHECKPOINT_REFS     = BASE_PATH + '/checkpoint_refs_v4.json'

# Model
MODEL = 'openai/gpt-oss-120b'

# Maximum characters per .md file sent to LLM (Pass 1 only)
MAX_CONTENT_CHARS = 60_000

# Save checkpoint every N processed rows
SAVE_EVERY = 5

print(f'Configuration loaded.')
print(f'INPUT_FILE  : {INPUT_FILE}')
print(f'OUTPUT_FILE : {OUTPUT_FILE}')
print(f'MODEL       : {MODEL}')

Configuration loaded.
INPUT_FILE  : /content/drive/My Drive/Mestrado/Mestrado PPCA/Dissertaçao/Artigos_C2_IC2Institute/ICCRTS/metadados_iccrts_v3.xlsx
OUTPUT_FILE : /content/drive/My Drive/Mestrado/Mestrado PPCA/Dissertaçao/Artigos_C2_IC2Institute/ICCRTS/metadados_iccrts_v4.xlsx
MODEL       : openai/gpt-oss-120b


In [5]:
# Cell 6: Load metadados_iccrts_v3.xlsx and initialize new columns
import pandas as pd

df = pd.read_excel(INPUT_FILE, engine='openpyxl')

# Initialize new columns if not already present
if 'LLM_KEYWORDS' not in df.columns:
    df['LLM_KEYWORDS'] = ''
if 'LLM_REFERENCES' not in df.columns:
    df['LLM_REFERENCES'] = ''

# Ensure KEYWORDS and REFERENCES are strings (handle NaN)
df['KEYWORDS']   = df['KEYWORDS'].fillna('')
df['REFERENCES'] = df['REFERENCES'].fillna('')
df['DRIVE_PATH'] = df['DRIVE_PATH'].fillna('')

print(f'DataFrame loaded: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
print()

# Summary of what needs processing
needs_keywords = (df['KEYWORDS'] == '').sum()
has_references = (df['REFERENCES'] != '').sum()
print(f'Pass 1 — Rows needing keywords (KEYWORDS empty)  : {needs_keywords}')
print(f'Pass 2 — Rows with references to format          : {has_references}')

DataFrame loaded: 488 rows, 14 columns
Columns: ['TITLE', 'AUTHORS', 'AUTHORS_APA', 'KEYWORDS', 'ABSTRACT', 'REFERENCES', 'CONFERENCE', 'YEAR', 'SOURCE_FILE', 'SOURCE_FOLDER', 'DRIVE_PATH', '_ERROR', 'LLM_KEYWORDS', 'LLM_REFERENCES']

Pass 1 — Rows needing keywords (KEYWORDS empty)  : 332
Pass 2 — Rows with references to format          : 483


In [6]:
# Cell 7: Prompt templates

# ── PASS 1: Keywords generation ──────────────────────────────────────────────

KEYWORDS_SYSTEM_PROMPT = """You are an expert scientific keyword extractor.
Given the full text of a scientific article in Markdown format, generate between 5 and 10 relevant academic keywords that best describe the article's core topics, methods, and contributions.

RULES:
1. Return keywords as semicolon-separated values.
2. Use established academic terminology.
3. Include both broad domain terms and specific technical terms.
4. Do not include author names or conference names.
5. Return ONLY valid JSON, no markdown code fences, no explanation.

OUTPUT FORMAT (strict JSON):
{"LLM_KEYWORDS": "keyword1; keyword2; keyword3; ..."}"""


def build_keywords_user_prompt(md_content: str) -> str:
    return f'Generate academic keywords for the following scientific article:\n\n---\n{md_content}\n---'


# ── PASS 2: Reference formatting ─────────────────────────────────────────────

REFS_SYSTEM_PROMPT = """You are an expert academic reference formatter.
You will receive a list of references separated by the pipe character "|".
Your task is to reformat each reference into APA 7th edition format.

RULES:
1. Standardize each reference to APA 7th edition.
2. If a reference is incomplete or ambiguous, format what is available.
3. Return each formatted reference separated by the pipe character "|".
4. Preserve the original order of references.
5. Do not add or remove references — only reformat.
6. Return ONLY valid JSON, no markdown code fences, no explanation.

OUTPUT FORMAT (strict JSON):
{"LLM_REFERENCES": "formatted_ref1|formatted_ref2|formatted_ref3|..."}"""


def build_refs_user_prompt(references_text: str) -> str:
    return f'Reformat the following references to APA 7th edition:\n\n{references_text}'


print('Prompt templates defined.')
print(f'Keywords system prompt length : {len(KEYWORDS_SYSTEM_PROMPT)} chars')
print(f'References system prompt length: {len(REFS_SYSTEM_PROMPT)} chars')

Prompt templates defined.
Keywords system prompt length : 606 chars
References system prompt length: 660 chars


In [7]:
# Cell 8: Generic LLM call function with retry logic and truncation for keywords
import json
import re
import time
from groq import Groq

# Instantiate Groq client
client = Groq(api_key=GROQ_API_KEY)


def truncate_for_keywords(content: str, max_chars: int) -> str:
    """
    Truncate markdown content for KEYWORD GENERATION only.
    
    Strategy: Head (80%) + Tail (20%) - NO references section needed.
    This is different from v2 extraction because:
    - We don't need references for keyword generation
    - References are already in the Excel file (to be reformatted in Pass 2)
    """
    if len(content) <= max_chars:
        return content

    original_len = len(content)

    # Simple head/tail truncation - no reference preservation needed for keywords
    head_size = int(max_chars * 0.8)
    tail_size = max_chars - head_size
    
    truncated = (
        content[:head_size]
        + '\n\n[...CONTENT TRUNCATED...]\n\n'
        + content[-tail_size:]
    )
    print(f'    [TRUNCATION] Keywords: {original_len:,} → {len(truncated):,} chars (head={head_size:,}, tail={tail_size:,})')
    return truncated


def call_llm(
    client: Groq,
    system_prompt: str,
    user_prompt: str,
    model: str = MODEL,
    max_retries: int = 3,
) -> dict:
    """
    Send a prompt to Groq LLM and parse the JSON response.
    Retries up to max_retries times on transient errors.
    Returns a dict with the parsed JSON, or a dict with '_ERROR' key on failure.
    """
    if not user_prompt.strip():
        return {'_ERROR': 'Empty user prompt'}

    for attempt in range(1, max_retries + 1):
        try:
            completion = client.chat.completions.create(
                model=model,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user',   'content': user_prompt},
                ],
                temperature=0.1,
                max_completion_tokens=8192,
                top_p=1,
                stream=False,
            )

            response_text = completion.choices[0].message.content.strip()

            # Strip potential markdown code fences (```json ... ```)
            if response_text.startswith('```'):
                lines = response_text.split('\n')
                response_text = '\n'.join(lines[1:-1]).strip()

            return json.loads(response_text)  # Success

        except json.JSONDecodeError as e:
            return {'_ERROR': f'JSON parse error: {e}'}

        except Exception as e:
            error_msg = str(e)
            print(f'  Attempt {attempt}/{max_retries} failed: {error_msg}')
            if attempt < max_retries:
                wait = 2 ** attempt  # Exponential backoff: 2s, 4s, 8s
                print(f'  Waiting {wait}s before retry...')
                time.sleep(wait)
            else:
                return {'_ERROR': error_msg}


print('call_llm() and smart_truncate() defined.')
print(f'Groq client ready (model: {MODEL}).')

call_llm() and smart_truncate() defined.
Groq client ready (model: openai/gpt-oss-120b).


In [8]:
# Cell 9: Checkpoint functions
import json
from pathlib import Path


def load_checkpoint(path: str) -> dict:
    """Load previously processed results. Returns dict keyed by row index string."""
    p = Path(path)
    if p.exists():
        with open(p, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f'Checkpoint loaded: {len(data)} records at {path}')
        return data
    print(f'No checkpoint found at {path}. Starting fresh.')
    return {}


def save_checkpoint(results: dict, path: str):
    """Persist the results dict to the checkpoint JSON file."""
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)


print('Checkpoint functions defined.')

Checkpoint functions defined.


In [9]:
# Cell 10: PASS 1 — Keywords Generation
# Reads .md files from Drive for articles with empty KEYWORDS
import time
from tqdm.notebook import tqdm

print('=' * 60)
print('PASS 1 — Keywords Generation')
print('=' * 60)

# Filter rows that need keywords
df_needs_kw = df[df['KEYWORDS'].str.strip() == ''].copy()
print(f'Articles needing keywords: {len(df_needs_kw)}')

# Load checkpoint
kw_results = load_checkpoint(CHECKPOINT_KEYWORDS)
newly_processed = 0

for idx, row in tqdm(df_needs_kw.iterrows(), total=len(df_needs_kw), desc='Pass 1 — Keywords'):
    row_key = str(idx)

    # Skip already processed
    if row_key in kw_results:
        continue

    drive_path = row.get('DRIVE_PATH', '')
    if not drive_path:
        print(f'  WARNING: No DRIVE_PATH for row {idx} ({row.get("SOURCE_FILE", "")}). Skipping.')
        kw_results[row_key] = {'_ERROR': 'No DRIVE_PATH'}
        continue

    # Read .md file from Drive
    try:
        with open(drive_path, 'r', encoding='utf-8') as f:
            md_content = f.read()
    except Exception as e:
        print(f'  ERROR reading {drive_path}: {e}')
        kw_results[row_key] = {'_ERROR': f'File read error: {e}'}
        newly_processed += 1
        if newly_processed % SAVE_EVERY == 0:
            save_checkpoint(kw_results, CHECKPOINT_KEYWORDS)
        time.sleep(1)
        continue

    # Truncate if needed
    md_content = truncate_for_keywords(md_content, MAX_CONTENT_CHARS)

    # Call LLM
    result = call_llm(
        client,
        KEYWORDS_SYSTEM_PROMPT,
        build_keywords_user_prompt(md_content),
    )

    kw_results[row_key] = result
    newly_processed += 1

    # Periodic checkpoint save
    if newly_processed % SAVE_EVERY == 0:
        save_checkpoint(kw_results, CHECKPOINT_KEYWORDS)
        print(f'  Checkpoint saved ({len(kw_results)} total records).')

    # Rate limiting
    time.sleep(1)

# Final checkpoint save
save_checkpoint(kw_results, CHECKPOINT_KEYWORDS)
print(f'\nPass 1 done! Total records in checkpoint: {len(kw_results)}')

PASS 1 — Keywords Generation
Articles needing keywords: 332
No checkpoint found at /content/drive/My Drive/Mestrado/Mestrado PPCA/Dissertaçao/Artigos_C2_IC2Institute/ICCRTS/checkpoint_keywords_v4.json. Starting fresh.


Pass 1 — Keywords:   0%|          | 0/332 [00:00<?, ?it/s]

    [TRUNCATION] Keywords: 60,491 → 60,029 chars (head=48,000, tail=12,000)
    [TRUNCATION] Keywords: 83,011 → 60,029 chars (head=48,000, tail=12,000)
  Checkpoint saved (5 total records).
    [TRUNCATION] Keywords: 122,414 → 60,029 chars (head=48,000, tail=12,000)
  Checkpoint saved (10 total records).
    [TRUNCATION] Keywords: 72,813 → 60,029 chars (head=48,000, tail=12,000)
  Checkpoint saved (15 total records).
    [TRUNCATION] Keywords: 75,790 → 60,029 chars (head=48,000, tail=12,000)
  Checkpoint saved (20 total records).
    [TRUNCATION] Keywords: 71,031 → 60,029 chars (head=48,000, tail=12,000)
  Checkpoint saved (25 total records).
    [TRUNCATION] Keywords: 62,970 → 60,029 chars (head=48,000, tail=12,000)
  Checkpoint saved (30 total records).
    [TRUNCATION] Keywords: 81,437 → 60,029 chars (head=48,000, tail=12,000)
  Checkpoint saved (35 total records).
    [TRUNCATION] Keywords: 64,399 → 60,029 chars (head=48,000, tail=12,000)
    [TRUNCATION] Keywords: 88,681 → 60,029 

In [10]:
# Cell 11: Apply Pass 1 results (keywords) to DataFrame

kw_applied = 0
kw_errors  = 0

for idx_str, result in kw_results.items():
    idx = int(idx_str)
    if '_ERROR' in result:
        kw_errors += 1
        continue
    kw_value = result.get('LLM_KEYWORDS', '')
    if kw_value:
        df.at[idx, 'LLM_KEYWORDS'] = kw_value
        kw_applied += 1

print(f'Pass 1 results applied to DataFrame:')
print(f'  Keywords generated successfully : {kw_applied}')
print(f'  Errors                          : {kw_errors}')
print(f'  LLM_KEYWORDS non-empty in df    : {(df["LLM_KEYWORDS"] != "").sum()}')

Pass 1 results applied to DataFrame:
  Keywords generated successfully : 332
  Errors                          : 0
  LLM_KEYWORDS non-empty in df    : 332


In [11]:
# Cell 12: PASS 2 — Reference Formatting
# Uses existing REFERENCES column from v3 — NO .md file reading needed
import time
from tqdm.notebook import tqdm

print('=' * 60)
print('PASS 2 — Reference Formatting (APA 7th edition)')
print('=' * 60)

# Filter rows that have non-empty REFERENCES
df_has_refs = df[df['REFERENCES'].str.strip() != ''].copy()
print(f'Articles with references to format: {len(df_has_refs)}')

# Load checkpoint
ref_results = load_checkpoint(CHECKPOINT_REFS)
newly_processed = 0

for idx, row in tqdm(df_has_refs.iterrows(), total=len(df_has_refs), desc='Pass 2 — References'):
    row_key = str(idx)

    # Skip already processed
    if row_key in ref_results:
        continue

    references_text = row['REFERENCES']
    if not references_text.strip():
        continue

    # Call LLM with REFERENCES column text directly (no .md file reading)
    result = call_llm(
        client,
        REFS_SYSTEM_PROMPT,
        build_refs_user_prompt(references_text),
    )

    ref_results[row_key] = result
    newly_processed += 1

    # Periodic checkpoint save
    if newly_processed % SAVE_EVERY == 0:
        save_checkpoint(ref_results, CHECKPOINT_REFS)
        print(f'  Checkpoint saved ({len(ref_results)} total records).')

    # Rate limiting
    time.sleep(1)

# Final checkpoint save
save_checkpoint(ref_results, CHECKPOINT_REFS)
print(f'\nPass 2 done! Total records in checkpoint: {len(ref_results)}')

PASS 2 — Reference Formatting (APA 7th edition)
Articles with references to format: 483
No checkpoint found at /content/drive/My Drive/Mestrado/Mestrado PPCA/Dissertaçao/Artigos_C2_IC2Institute/ICCRTS/checkpoint_refs_v4.json. Starting fresh.


Pass 2 — References:   0%|          | 0/483 [00:00<?, ?it/s]

  Checkpoint saved (5 total records).
  Checkpoint saved (10 total records).
  Checkpoint saved (15 total records).
  Checkpoint saved (20 total records).
  Checkpoint saved (25 total records).
  Checkpoint saved (30 total records).
  Checkpoint saved (35 total records).
  Checkpoint saved (40 total records).
  Checkpoint saved (45 total records).
  Checkpoint saved (50 total records).
  Checkpoint saved (55 total records).
  Checkpoint saved (60 total records).
  Checkpoint saved (65 total records).
  Checkpoint saved (70 total records).
  Checkpoint saved (75 total records).
  Checkpoint saved (80 total records).
  Checkpoint saved (85 total records).
  Checkpoint saved (90 total records).
  Checkpoint saved (95 total records).
  Checkpoint saved (100 total records).
  Checkpoint saved (105 total records).
  Checkpoint saved (110 total records).
  Checkpoint saved (115 total records).
  Checkpoint saved (120 total records).
  Checkpoint saved (125 total records).
  Checkpoint saved (

In [12]:
# Cell 13: Apply Pass 2 results (formatted references) to DataFrame

ref_applied = 0
ref_errors  = 0

for idx_str, result in ref_results.items():
    idx = int(idx_str)
    if '_ERROR' in result:
        ref_errors += 1
        continue
    ref_value = result.get('LLM_REFERENCES', '')
    if ref_value:
        df.at[idx, 'LLM_REFERENCES'] = ref_value
        ref_applied += 1

print(f'Pass 2 results applied to DataFrame:')
print(f'  References formatted successfully : {ref_applied}')
print(f'  Errors                            : {ref_errors}')
print(f'  LLM_REFERENCES non-empty in df    : {(df["LLM_REFERENCES"] != "").sum()}')

Pass 2 results applied to DataFrame:
  References formatted successfully : 427
  Errors                            : 56
  LLM_REFERENCES non-empty in df    : 427


In [13]:
# Cell 14: Export to metadados_iccrts_v4.xlsx
from pathlib import Path

# Desired column order — new columns added after REFERENCES
col_order = [
    'TITLE', 'AUTHORS', 'AUTHORS_APA', 'KEYWORDS', 'LLM_KEYWORDS',
    'ABSTRACT', 'REFERENCES', 'LLM_REFERENCES',
    'CONFERENCE', 'YEAR', 'SOURCE_FILE', 'SOURCE_FOLDER', 'DRIVE_PATH',
]
existing_cols = [c for c in col_order if c in df.columns]
extra_cols    = [c for c in df.columns if c not in col_order]
df_export = df[existing_cols + extra_cols]

df_export.to_excel(OUTPUT_FILE, index=False, engine='openpyxl')

print(f'DataFrame shape : {df_export.shape}')
print(f'Columns         : {list(df_export.columns)}')
print(f'Excel saved → {OUTPUT_FILE}')

DataFrame shape : (488, 14)
Columns         : ['TITLE', 'AUTHORS', 'AUTHORS_APA', 'KEYWORDS', 'LLM_KEYWORDS', 'ABSTRACT', 'REFERENCES', 'LLM_REFERENCES', 'CONFERENCE', 'YEAR', 'SOURCE_FILE', 'SOURCE_FOLDER', 'DRIVE_PATH', '_ERROR']
Excel saved → /content/drive/My Drive/Mestrado/Mestrado PPCA/Dissertaçao/Artigos_C2_IC2Institute/ICCRTS/metadados_iccrts_v4.xlsx


In [14]:
# Cell 15: Quality summary and sample inspection
import pandas as pd

print('=' * 60)
print('ENRICHMENT SUMMARY — v4')
print('=' * 60)
print(f'Total articles in dataset : {len(df_export)}')
print()

# Keywords coverage
orig_kw_filled  = (df_export['KEYWORDS'].str.strip() != '').sum()
llm_kw_filled   = (df_export['LLM_KEYWORDS'].str.strip() != '').sum()
total_kw_filled = ((df_export['KEYWORDS'].str.strip() != '') | (df_export['LLM_KEYWORDS'].str.strip() != '')).sum()
print(f'Keywords coverage:')
print(f'  Original KEYWORDS filled    : {orig_kw_filled} ({orig_kw_filled/len(df_export)*100:.1f}%)')
print(f'  LLM_KEYWORDS generated      : {llm_kw_filled} ({llm_kw_filled/len(df_export)*100:.1f}%)')
print(f'  Total with any keywords     : {total_kw_filled} ({total_kw_filled/len(df_export)*100:.1f}%)')
print()

# References coverage
orig_ref_filled = (df_export['REFERENCES'].str.strip() != '').sum()
llm_ref_filled  = (df_export['LLM_REFERENCES'].str.strip() != '').sum()
print(f'References coverage:')
print(f'  Original REFERENCES filled  : {orig_ref_filled} ({orig_ref_filled/len(df_export)*100:.1f}%)')
print(f'  LLM_REFERENCES formatted    : {llm_ref_filled} ({llm_ref_filled/len(df_export)*100:.1f}%)')
print()

# Error summary
kw_err_count  = sum(1 for v in kw_results.values()  if '_ERROR' in v)
ref_err_count = sum(1 for v in ref_results.values() if '_ERROR' in v)
print(f'Errors:')
print(f'  Pass 1 (keywords) errors    : {kw_err_count}')
print(f'  Pass 2 (references) errors  : {ref_err_count}')
print('=' * 60)

# Sample: first 3 generated keywords
print('\n--- Sample: LLM_KEYWORDS (first 3 generated) ---')
sample_kw = df_export[df_export['LLM_KEYWORDS'].str.strip() != ''].head(3)
for _, row in sample_kw.iterrows():
    print(f'\n  [{row.get("SOURCE_FILE", "")}]')
    print(f'  Original KEYWORDS  : {row["KEYWORDS"][:100] if row["KEYWORDS"] else "(empty)"}')
    print(f'  LLM_KEYWORDS       : {row["LLM_KEYWORDS"][:150]}')

# Sample: first 2 formatted references
print('\n--- Sample: LLM_REFERENCES (first 2 formatted) ---')
sample_ref = df_export[df_export['LLM_REFERENCES'].str.strip() != ''].head(2)
for _, row in sample_ref.iterrows():
    print(f'\n  [{row.get("SOURCE_FILE", "")}]')
    orig_refs = row['REFERENCES'].split('|')[:2] if row['REFERENCES'] else []
    llm_refs  = row['LLM_REFERENCES'].split('|')[:2] if row['LLM_REFERENCES'] else []
    print(f'  Original (first 2) :')
    for r in orig_refs:
        print(f'    {r[:120]}')
    print(f'  Formatted (first 2):')
    for r in llm_refs:
        print(f'    {r[:120]}')

ENRICHMENT SUMMARY — v4
Total articles in dataset : 488

Keywords coverage:
  Original KEYWORDS filled    : 156 (32.0%)
  LLM_KEYWORDS generated      : 332 (68.0%)
  Total with any keywords     : 488 (100.0%)

References coverage:
  Original REFERENCES filled  : 483 (99.0%)
  LLM_REFERENCES formatted    : 427 (87.5%)

Errors:
  Pass 1 (keywords) errors    : 0
  Pass 2 (references) errors  : 56

--- Sample: LLM_KEYWORDS (first 3 generated) ---

  [21th_ICCRTS_paper_10.md]
  Original KEYWORDS  : (empty)
  LLM_KEYWORDS       : cyberspace network mapping; cyber protection teams; cognitive engineering; human factors evaluation; tool usability and utility; network topology disc

  [21th_ICCRTS_paper_17.md]
  Original KEYWORDS  : (empty)
  LLM_KEYWORDS       : Command and Control (C2); Mobile Command; Dispersed Operations; Collaborative Mapping; Situational Awareness Tools; Simulation-based Experimentation; 

  [21th_ICCRTS_paper_20.md]
  Original KEYWORDS  : (empty)
  LLM_KEYWORDS       : se